# Weed detection under domain shift: Kaggle training

Runs `src/train.py` for one job (a list of runs) and collects the run folders in `/kaggle/working/runs` (downloadable as notebook output).

**Setup (once):**
1. *Add Input* → your private dataset with the contents of `weed-ds-yolo.zip` (made by `src/package_for_kaggle.py`).
2. Code is cloned from the public GitHub repo (`REPO_URL`); push code changes before running.
3. *Settings*: Accelerator = GPU T4 x2, Internet = on (for `pip` and pretrained weights).

**Jobs:** `pilot` (5-epoch timing run per model), `baseline` (both models × 3 seeds), `check` (RT-DETR 200-epoch convergence check + YOLO ablations), `ablation` (RT-DETR: imgsz 1024, HSV off, HSV strong × 3 seeds), `hue` (full hue randomisation; both models × 3 seeds). With two GPUs, runs are spread over both and run in parallel; each run's full log goes to `/kaggle/working/logs/`.

**To run unattended:** *Save Version* → *Save & Run All (Commit)*. Afterwards download `runs.zip` from the version's Output tab and unzip it into the repo's `runs/`.

In [ ]:
REPO_URL = "https://github.com/gbudding/weed-detection-domain-shift"   # empty = use an attached code dataset instead
JOB = "hue"   # "pilot", "baseline", "check", "ablation" or "hue"


In [ ]:
import glob, json, os, shutil, subprocess, sys
from pathlib import Path

# Data: copy to a writable location (Ultralytics writes a label cache next to the labels).
# The dataset may be mounted unpacked (greenhouse.yaml somewhere below /kaggle/input) or as the zip itself.
DATA = Path("/tmp/yolo")
if not DATA.exists():
    data_yaml = glob.glob("/kaggle/input/**/greenhouse.yaml", recursive=True)
    data_zip = glob.glob("/kaggle/input/**/weed-ds-yolo.zip", recursive=True)
    if data_yaml:
        shutil.copytree(Path(data_yaml[0]).parent, DATA)
    elif data_zip:
        shutil.unpack_archive(data_zip[0], DATA)
    else:
        listing = subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True).stdout
        raise FileNotFoundError("No greenhouse.yaml or weed-ds-yolo.zip under /kaggle/input. "
                                "Is the dataset added as input? Contents:" + chr(10) + (listing or "(empty)"))

# Code: clone the repo, or use the attached code dataset.
CODE = Path("/tmp/code")
if not (CODE / "src/train.py").exists():
    shutil.rmtree(CODE, ignore_errors=True)  # leftovers of an interrupted clone
    if REPO_URL:
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True, timeout=120,
                           env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as e:
            raise RuntimeError("git clone failed. Is Internet switched on (Settings > Internet; "
                               "needs a phone-verified account)?") from e
    else:
        train_py = glob.glob("/kaggle/input/**/src/train.py", recursive=True)
        assert train_py, "No REPO_URL and no code dataset with src/train.py attached"
        shutil.copytree(Path(train_py[0]).parents[1], CODE)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(len(list((DATA / "images").glob("*.jpg"))), "images in", DATA)

In [ ]:
import threading, time
import torch

# One dict per run; keys are train.py arguments (without the leading --).
SEEDS = (0, 1, 2)
BATCH = {"yolo11s": 16, "rtdetr-l": 8}
ABLATIONS = [dict(imgsz=1024), dict(hsv="off"), dict(hsv="strong")]
if JOB == "pilot":
    RUNS = [dict(model=m, epochs=5, batch=b, name=f"pilot_{m}") for m, b in BATCH.items()]
elif JOB == "baseline":
    RUNS = [dict(model=m, seed=s, batch=b, epochs=100) for m, b in BATCH.items() for s in SEEDS]
elif JOB == "ablation":
    # RT-DETR ablations (the YOLO ablations ran in "check"). At imgsz 1024 RT-DETR needs ~17 GB at batch 8
    # (6.7 GB measured at 640), more than a T4 has, so it uses batch 4 there.
    RUNS = [dict(model="rtdetr-l", seed=s, batch=4 if a.get("imgsz") == 1024 else BATCH["rtdetr-l"], epochs=100, **a)
            for a in ABLATIONS for s in SEEDS]
elif JOB == "check":
    # RT-DETR convergence check (200 epochs, seed 0) next to the YOLO ablations (YOLO stays at 100 epochs).
    RUNS = [dict(model="rtdetr-l", seed=0, batch=BATCH["rtdetr-l"], epochs=200, name="rtdetr-l_img640_hsv-default_s0_ep200")]
    RUNS += [dict(model="yolo11s", seed=s, batch=BATCH["yolo11s"], epochs=100, **a) for a in ABLATIONS for s in SEEDS]
elif JOB == "hue":
    # Full hue randomisation (src/train.py HSV "hue"), compared with "strong". Remove "rtdetr-l" for YOLO only.
    RUNS = [dict(model=m, seed=s, batch=BATCH[m], epochs=100, hsv="hue") for m in ("yolo11s", "rtdetr-l") for s in SEEDS]
else:
    raise ValueError(JOB)


def cost(run):
    # Relative run time from the pilot: RT-DETR ~4x YOLO, imgsz 1024 ~2.5x imgsz 640, linear in epochs.
    return (4 if run["model"].startswith("rtdetr") else 1) * (run.get("imgsz", 640) / 640) ** 2 * run.get("epochs", 100) / 100


# Spread runs over the GPUs, longest first, each to the GPU with the least work so far.
n_gpu = max(1, torch.cuda.device_count())
queues, load = [[] for _ in range(n_gpu)], [0.0] * n_gpu
for run in sorted(RUNS, key=cost, reverse=True):
    g = load.index(min(load))
    queues[g].append(run)
    load[g] += cost(run)

OUT, LOGS = Path("/kaggle/working/runs"), Path("/kaggle/working/logs")
LOGS.mkdir(parents=True, exist_ok=True)


def worker(gpu, runs):
    # Each GPU gets its own copy of the dataset, so parallel runs never share a label cache file.
    data = Path(f"/tmp/yolo_gpu{gpu}")
    if not data.exists():
        shutil.copytree(DATA, data)
    for run in runs:
        name = run.get("name") or f"{run['model']}_img{run.get('imgsz', 640)}_hsv-{run.get('hsv', 'default')}_s{run.get('seed', 0)}"
        cmd = [sys.executable, str(CODE / "src/train.py"), "--data-dir", str(data), "--project", str(OUT),
               "--device", str(gpu), "--workers", "2", "--name", name]
        cmd += [x for k, v in run.items() if k != "name" for x in (f"--{k}", str(v))]
        print(f"[gpu{gpu}] start {name}", flush=True)
        t0 = time.time()
        with open(LOGS / f"{name}.log", "w") as log:
            # A failing run is logged but does not stop the remaining runs.
            rc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT).returncode
        print(f"[gpu{gpu}] done  {name}: exit code {rc}, {(time.time() - t0) / 60:.1f} min", flush=True)
        if rc != 0:
            print(open(LOGS / f"{name}.log").read()[-3000:], flush=True)


print(f"{len(RUNS)} runs on {n_gpu} GPU(s):", [[r.get("model") for r in q] for q in queues])
threads = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues)]
for t in threads:
    t.start()
for t in threads:
    t.join()

In [ ]:
rows = []
for d in sorted(OUT.glob("*/")):
    if (d / "metrics.json").exists():
        cfg, m = json.loads((d / "run_config.json").read_text()), json.loads((d / "metrics.json").read_text())
        rows.append({"run": d.name, "gpu": cfg["gpu"], "epochs": cfg["epochs"], "batch": cfg["batch"],
                     "gpu_mem_gb": cfg.get("gpu_mem_peak_gb"),
                     "train_min": round(cfg["train_seconds"] / 60, 1),
                     "min_per_epoch": round(cfg["train_seconds"] / 60 / cfg["epochs"], 2),
                     **{f"{t}_mAP50": round(m[t]["mAP50"], 3) for t in m}})
    else:
        rows.append({"run": d.name, "error": "no metrics.json"})
import pandas as pd
display(pd.DataFrame(rows))

# Zip run folders and logs without last.pt (only needed to resume training; best.pt is kept).
for p in OUT.glob("*/weights/last.pt"):
    p.unlink()
shutil.copytree(LOGS, OUT / "_logs", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/runs", "zip", OUT)
print(f"/kaggle/working/runs.zip: {os.path.getsize('/kaggle/working/runs.zip') / 1e6:.0f} MB")